In [8]:
import pandas as pd
import plotly.express as px


dane = pd.read_csv("data1.csv")
dane

fig1 = px.scatter(dane,x="x1",y="y",title="x1 vs y")
fig2 = px.scatter(dane,x="x2",y="y",title="x2 vs y")
fig3 = px.histogram(dane,x="y")
fig1.show()
fig2.show()
fig3.show()

widać jakies liniowe ustawienie wartości x1 i x2 względem y, jak y rosnie to x1 rosnie a x2 maleje
plus jest jakaś wartosc ekstremalna, y=805

In [9]:
correlation = dane.corr()
correlation

,x1,x2,y
x1,1.000000,-0.103316,0.780708
x2,-0.103316,1.000000,-0.440934
y,0.780708,-0.440934,1.000000


widzimy że x1 jest dość mocno skorelowane względem y, a x2 tylko trochę, ale to potwierdza to co widać na wykresach, y rosnie to x1 tez, y rosnie to x2 troche maleje, a jednoczesnie x1 i x2 nie są ze sobą skorelowane co eliminuje nam problem współliniowości

In [10]:
dane["diff"]=dane["x1"]-dane["x2"]
fig3 = px.scatter(dane,x="diff",y="y",title = " x1-x2 vs y")
fig3.show()

Widzimy,że wiekszosc danych y to ok. x1 - x2 oprócz jednego outliera, który bedzie nam psuł regresje liniową

In [11]:
odchyl = dane["diff"]-dane["y"]
odchyl

0    -7.319975
1    -8.232254
2    -8.293072
3    -7.285649
4    -9.865775
        ...   
95   -7.346671
96   -9.765454
97   -8.404982
98   -6.739116
99   -8.917862
Length: 100, dtype: float64

In [12]:
def preprocess_data(y_train,X_train):
    odchylenie = y_train - (X_train["x1"]-X_train["x2"])
    maska = abs(odchylenie)<=20
    return y_train[maska],X_train[maska]

In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor

X = dane[["x1","x2"]]
y= dane["y"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
y_train_proc,X_train_proc=preprocess_data(y_train,X_train)

baseline = DummyRegressor(strategy="mean")
linear_regression = LinearRegression()

baseline.fit(X_train,y_train)
y_pred_baseline =baseline.predict(X_test)
mse_baseline = mean_squared_error(y_test,y_pred_baseline)

linear_regression.fit(X_train,y_train)
y_pred_lr = linear_regression.predict(X_test)
mse_lr = mean_squared_error(y_test,y_pred_lr)


baseline.fit(X_train_proc,y_train_proc)
y_pred_baseline_proc = baseline.predict(X_test)
mse_baseline_proc = mean_squared_error(y_test,y_pred_baseline_proc)

linear_regression.fit(X_train_proc,y_train_proc)
y_pred_lr_proc = linear_regression.predict(X_test)
mse_lr_proc = mean_squared_error(y_test,y_pred_lr_proc)


table_data = {
    "Model" : ["Baseline","Baseline","Regresja Liniowa", "Regresja Liniowa"],
    "Dane Treningowe" : ["Surowe","Oczyszczone","Surowe","Oczyszczone"],
    "Błąd MSE" : [mse_baseline,mse_baseline_proc,mse_lr,mse_lr_proc]
}
table = pd.DataFrame(table_data)
print(table)

linear_regression.coef_



              Model Dane Treningowe      Błąd MSE
0          Baseline          Surowe  70092.425271
1          Baseline     Oczyszczone  71561.278594
2  Regresja Liniowa          Surowe   1043.043946
3  Regresja Liniowa     Oczyszczone      0.719293


array([ 1.00044086, -0.99881687])